<a href="https://colab.research.google.com/github/JessemanGray/Syn-Aesthetics/blob/main/ChillsDB_notebook.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [1]:
# src/rag_agent/data_processor.py
import pandas as pd
import numpy as np
import requests
import io
import plotly.graph_objects as go
import networkx as nx
from scipy.spatial import KDTree
from sklearn.preprocessing import MinMaxScaler

class SynAestheticsDataProcessor:
    def __init__(self):
        self.df_raw = None
        self.df_processed = None
        self.graph = None

    def load_chills_data(self):
        """Load ChillsDB 2.0 from Figshare"""
        article_id = 23935611
        api_url = f"https://api.figshare.com/v2/articles/{article_id}/files"

        print("📥 Fetching file list from Figshare...")
        response = requests.get(api_url, headers={"User-Agent": "Mozilla/5.0"})
        response.raise_for_status()
        files = response.json()

        data_file = next((f for f in files if f['name'].endswith('.csv')), None)
        if data_file is None:
            raise ValueError("No CSV file found")

        print(f"   Downloading: {data_file['name']}")
        download_response = requests.get(data_file['download_url'], headers={"User-Agent": "Mozilla/5.0"})
        download_response.raise_for_status()

        self.df_raw = pd.read_csv(io.StringIO(download_response.content.decode('utf-8')))
        print(f"✅ Loaded {len(self.df_raw)} records")
        return self.df_raw

    def process_chills_data(self):
        df = self.df_raw.copy()

        if 'Chills Intensity' in df.columns:
            df['Chill Rating'] = (df['Chills Intensity'].fillna(0) / 10).round(1).clip(0, 10)
        else:
            df['Chill Rating'] = np.random.uniform(0, 10, len(df)).round(1)

        if 'Valence Shift' in df.columns:
            max_val = df['Valence Shift'].abs().max()
            df['Valence Score'] = df['Valence Shift'] / max_val if max_val > 0 else 0
        else:
            df['Valence Score'] = np.random.uniform(-1, 1, len(df))

        def get_effect(row):
            intensity = row.get('Chills Intensity', 0)
            valence = row.get('Valence Score', 0)
            if pd.isna(intensity) or intensity == 0:
                return 'neutral'
            if valence > 0.1:
                return 'beneficial'
            elif valence < -0.1:
                return 'dischordant'
            return 'neutral'

        df['Chill Effect'] = df.apply(get_effect, axis=1)
        self.df_processed = df
        print(f"✅ Processed {len(df)} records")
        return self.df_processed

    def build_phyllotactic_kg(self):
        df = self.df_processed.sample(min(len(self.df_processed), 5000), random_state=42).reset_index(drop=True)

        scaler = MinMaxScaler()
        features = scaler.fit_transform(df[['Chill Rating', 'Liking']].fillna(0))

        golden_angle = np.pi * (3 - np.sqrt(5))
        radius = 25
        x_vals, y_vals, z_vals = [], [], []
        for i in range(len(df)):
            theta = i * golden_angle
            phi = np.arccos(1 - 2 * (i + 0.5) / len(df))
            r = radius * (0.5 + 0.5 * features[i, 0])
            x_vals.append(r * np.sin(phi) * np.cos(theta))
            y_vals.append(r * np.sin(phi) * np.sin(theta))
            z_vals.append(r * np.cos(phi))

        df['kg_x'], df['kg_y'], df['kg_z'] = x_vals, y_vals, z_vals

        def get_color(valence, intensity):
            v_norm = (valence + 1) / 2
            r = int(255 * (1 - v_norm))
            g = 0
            b = int(255 * v_norm)
            brightness = 0.6 + 0.4 * intensity
            return f'rgb({min(255,int(r*brightness))}, {min(255,int(g*brightness))}, {min(255,int(b*brightness))})'

        df['color'] = df.apply(
            lambda row: get_color(row['Valence Score'], row.get('Chills Intensity', 50) / 100),
            axis=1
        )

        G = nx.Graph()
        for _, row in df.iterrows():
            G.add_node(
                str(row.name),
                pos=(row['kg_x'], row['kg_y'], row['kg_z']),
                color=row['color'],
                rating=row['Chill Rating'],
                effect=row['Chill Effect'],
                valence=row['Valence Score']
            )

        positions = df[['kg_x', 'kg_y', 'kg_z']].values
        tree = KDTree(positions)
        for i in range(len(df)):
            dist, idx = tree.query(positions[i], k=5)
            for j in idx[1:]:
                if i != j and dist[1] < 12:
                    G.add_edge(str(i), str(j))

        self.graph = G
        print(f"✅ KG: {G.number_of_nodes()} nodes, {G.number_of_edges()} edges")
        return G

    def render_kg_viz(self):
        if self.graph is None:
            self.build_phyllotactic_kg()

        G = self.graph

        edge_x, edge_y, edge_z = [], [], []
        for u, v in G.edges():
            x0, y0, z0 = G.nodes[u]['pos']
            x1, y1, z1 = G.nodes[v]['pos']
            edge_x.extend([x0, x1, None])
            edge_y.extend([y0, y1, None])
            edge_z.extend([z0, z1, None])

        node_x = [G.nodes[n]['pos'][0] for n in G.nodes]
        node_y = [G.nodes[n]['pos'][1] for n in G.nodes]
        node_z = [G.nodes[n]['pos'][2] for n in G.nodes]
        node_colors = [G.nodes[n]['color'] for n in G.nodes]

        hover_text = [
            f"Rating: {G.nodes[n]['rating']:.1f}<br>Effect: {G.nodes[n]['effect']}<br>Valence: {G.nodes[n]['valence']:.2f}"
            for n in G.nodes
        ]

        fig = go.Figure()

        fig.add_trace(go.Scatter3d(
            x=edge_x, y=edge_y, z=edge_z,
            mode='lines',
            line=dict(color='rgba(128,128,128,0.06)', width=0.5),
            hoverinfo='none',
            showlegend=False
        ))

        fig.add_trace(go.Scatter3d(
            x=node_x, y=node_y, z=node_z,
            mode='markers',
            marker=dict(size=9, color=node_colors, opacity=0.9, line=dict(width=0)),
            text=hover_text,
            hoverinfo='text',
            showlegend=False
        ))

        fig.update_layout(
            scene=dict(
                xaxis=dict(visible=False, showgrid=False, showbackground=False, zeroline=False),
                yaxis=dict(visible=False, showgrid=False, showbackground=False, zeroline=False),
                zaxis=dict(visible=False, showgrid=False, showbackground=False, zeroline=False),
                bgcolor='black',
                aspectmode='cube'
            ),
            paper_bgcolor='black',
            plot_bgcolor='black',
            margin=dict(l=0, r=0, b=0, t=0),
            hoverlabel=dict(bgcolor='black', font=dict(color='white', size=11)),
            showlegend=False
        )

        fig.update_layout(scene_camera=dict(eye=dict(x=2.0, y=2.0, z=1.0), center=dict(x=0, y=0, z=0)))
        return fig

    def run_pipeline(self):
        self.load_chills_data()
        self.process_chills_data()
        self.build_phyllotactic_kg()
        return self.render_kg_viz()

if __name__ == "__main__":
    processor = SynAestheticsDataProcessor()
    fig = processor.run_pipeline()
    fig.show()  # <-- ADD THIS LINE
    fig.write_html("chillsdb2_sphere.html")
    print("✅ Saved to chillsdb2_sphere.html")


📥 Fetching file list from Figshare...
   Downloading: ChillsDB 2 - ChillsDB 2.csv
✅ Loaded 2937 records
✅ Processed 2937 records
✅ KG: 2937 nodes, 7217 edges


✅ Saved to chillsdb2_sphere.html
